In [1]:
#### mv utils ####
import pyg_modules.data as d
import pyg_modules.utils as u
from pyg_modules.utils import vprint, dict_summary

#### packages ####
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from pathlib import Path
from sklearn.preprocessing import RobustScaler
from torch_geometric.data import InMemoryDataset, Data

#### typing ####
from numpy import ndarray
from pandas import DataFrame, Series
from torch import Tensor
from typing import Literal, Union

In [2]:
#### init data ####

# dataset dir
datasets = Path('/home/mv18gs/Documents/GitHub/pathway_model/datasets/')

# get device
device, generator = u.Devices().auto_set_device(['cuda:1', 'cuda:0'])

*** Device() ***
device = cuda:2



---

In [3]:
class KEGGPathways():
    def __init__(self, filepath:str):
        self.data = pd.read_csv(filepath)
        self.unique_ensg = pd.concat([self.data['ensembl1'], self.data['ensembl2']]).unique().tolist()

In [4]:
kegg = KEGGPathways(datasets/'other'/'relation_ohe.csv')
kegg.data

,pathway_name,ensembl1,ensembl2,activation,binding/association,compound,dephosphorylation,dissociation,expression,glycosylation,indirect,indirect effect,inhibition,methylation,missing interaction,phosphorylation,repression,state change,ubiquitination
0,path:hsa04975,ENSG00000204310,ENSG00000067113,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False
1,path:hsa04975,ENSG00000204310,ENSG00000141934,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False
2,path:hsa04975,ENSG00000204310,ENSG00000162407,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False
3,path:hsa04975,ENSG00000169692,ENSG00000067113,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False
4,path:hsa04975,ENSG00000169692,ENSG00000141934,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75934,path:hsa00190,ENSG00000136888,ENSG00000180817,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False
75935,path:hsa00190,ENSG00000136888,ENSG00000107902,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False
75936,path:hsa00190,ENSG00000241468,ENSG00000138777,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False
75937,path:hsa00190,ENSG00000241468,ENSG00000180817,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False


---

In [5]:
def handle_log_zeros(
    x: Union[ndarray, DataFrame, Series, Tensor, float, int], 
    method: Literal['replace','log1p'] = 'log1p'
) -> Union[ndarray, DataFrame, Series, Tensor, float, int]:  
    '''
    Handle zero values in data, for use in transforms such as np.log()
    '''
    # replace 0 with small number
    if method == 'replace':
        if isinstance(x, (pd.DataFrame, pd.Series)):
            x = x.replace(0, 1e-6)
        elif isinstance(x, torch.Tensor):
            x = torch.where(x == 0, torch.tensor(1e-6, dtype=x.dtype, device=x.device), x)
        else:  # np.ndarray or numeric
            x = np.where(x == 0, 1e-6, x)

    # log1p method (add 1 before log)
    elif method == 'log1p':
        x = x + 1

    # error msg
    else:
        print("Invalid method; adjust_zeros method not applied. Use 'replace' or 'log1p'.")

    return x

In [6]:
class TCGAGeneCounts():
    def __init__(self, tcga_project:str, tcga_dir:str):
        # read counts
        self.data = self._read(tcga_project, tcga_dir)

    def _read(self, tcga_project:str, tcga_dir:str) -> DataFrame:
        # read df
        df = pd.read_csv(Path(tcga_dir) / f'{tcga_project}_gene_counts.csv')

        # rename 'Unnamed: 0' col if applicable
        if 'Unnamed: 0' in df.columns:
            df = df.rename(columns={'Unnamed: 0':'ensg'})

        # remove ensg version
        df['ensg'] = df['ensg'].str.split('.').str[0]

        # set index, column name
        df = df.set_index('ensg')
        df.columns.name = 'barcode'

        return df.T
    
    def _scale(self, gene_counts:pd.DataFrame, scale_method:Literal['robust', None]):
        # return if none
        if scale_method == None:
            return gene_counts

        # select scaler        
        if scale_method == 'robust':
            scaler = RobustScaler()

        # scale data
        scaled_values = scaler.fit_transform(gene_counts.T).T

        # convert to df
        gene_counts = pd.DataFrame(
            scaled_values,
            index=gene_counts.index,
            columns=gene_counts.columns
        )

        return gene_counts

In [7]:
brca = TCGAGeneCounts(
    tcga_project='TCGA-BRCA',
    tcga_dir=datasets/'tcga',
)

In [10]:
x = brca.data[kegg.unique_ensg]

In [17]:
x

ensg,ENSG00000204310,ENSG00000169692,ENSG00000067113,ENSG00000141934,ENSG00000162407,ENSG00000166391,ENSG00000146535,ENSG00000172380,ENSG00000109756,ENSG00000058335,...,ENSG00000007312,ENSG00000063854,ENSG00000119640,ENSG00000170634,ENSG00000196531,ENSG00000198648,ENSG00000172939,ENSG00000138777,ENSG00000180817,ENSG00000107902
barcode,,,,,,,,,,,,,,,,,,,,,
TCGA-A2-A25D-01A-12R-A16F-07,2109,1497,1456,2568,2518,724,2768,4570,1754,82,...,354,2246,318,446,38833,4085,3681,5435,10602,1007
TCGA-BH-A201-01A-11R-A14M-07,1771,1750,2591,2628,6482,245,5808,14684,3420,112,...,72,1046,331,573,27914,4547,5156,4971,11470,1116
TCGA-AC-A23C-01A-12R-A169-07,3148,3082,1588,7714,2934,982,8443,13434,3354,79,...,90,1903,352,913,24630,6019,5541,5612,6950,1398
TCGA-AR-A5QP-01A-11R-A28M-07,1469,2419,1713,2810,2503,34,3795,7107,1480,57,...,140,1889,311,705,25217,4286,1778,4395,6697,1022
TCGA-C8-A12P-01A-11R-A115-07,1626,3922,819,4935,2430,653,5548,6680,1690,110,...,43,2615,291,141,17224,251,2266,1226,19317,1107
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TCGA-LL-A5YP-01A-21R-A28M-07,2195,2348,1365,3254,3187,13,3473,3087,1859,104,...,160,1917,166,670,10342,865,4708,3024,8455,1738
TCGA-AO-A03L-01A-41R-A056-07,2473,3906,1410,10382,3313,17,4520,5050,2093,71,...,71,2333,354,212,35356,6047,2380,4209,14286,707
TCGA-BH-A42T-01A-11R-A24H-07,3261,6990,1629,18509,1779,88,5300,3006,1379,22,...,109,4590,319,704,37013,2204,1918,3961,10618,3196


In [20]:
import seaborn as sns

In [40]:
x

ensg,ENSG00000204310,ENSG00000169692,ENSG00000067113,ENSG00000141934,ENSG00000162407,ENSG00000166391,ENSG00000146535,ENSG00000172380,ENSG00000109756,ENSG00000058335,...,ENSG00000007312,ENSG00000063854,ENSG00000119640,ENSG00000170634,ENSG00000196531,ENSG00000198648,ENSG00000172939,ENSG00000138777,ENSG00000180817,ENSG00000107902
barcode,,,,,,,,,,,,,,,,,,,,,
TCGA-A2-A25D-01A-12R-A16F-07,2109,1497,1456,2568,2518,724,2768,4570,1754,82,...,354,2246,318,446,38833,4085,3681,5435,10602,1007
TCGA-BH-A201-01A-11R-A14M-07,1771,1750,2591,2628,6482,245,5808,14684,3420,112,...,72,1046,331,573,27914,4547,5156,4971,11470,1116
TCGA-AC-A23C-01A-12R-A169-07,3148,3082,1588,7714,2934,982,8443,13434,3354,79,...,90,1903,352,913,24630,6019,5541,5612,6950,1398
TCGA-AR-A5QP-01A-11R-A28M-07,1469,2419,1713,2810,2503,34,3795,7107,1480,57,...,140,1889,311,705,25217,4286,1778,4395,6697,1022
TCGA-C8-A12P-01A-11R-A115-07,1626,3922,819,4935,2430,653,5548,6680,1690,110,...,43,2615,291,141,17224,251,2266,1226,19317,1107
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TCGA-LL-A5YP-01A-21R-A28M-07,2195,2348,1365,3254,3187,13,3473,3087,1859,104,...,160,1917,166,670,10342,865,4708,3024,8455,1738
TCGA-AO-A03L-01A-41R-A056-07,2473,3906,1410,10382,3313,17,4520,5050,2093,71,...,71,2333,354,212,35356,6047,2380,4209,14286,707
TCGA-BH-A42T-01A-11R-A24H-07,3261,6990,1629,18509,1779,88,5300,3006,1379,22,...,109,4590,319,704,37013,2204,1918,3961,10618,3196


In [31]:
x.describe().T

,count,mean,std,min,25%,50%,75%,max
ensg,,,,,,,,
ENSG00000204310,1231.0,2113.142973,930.747629,127.0,1502.0,1972.0,2527.5,7095.0
ENSG00000169692,1231.0,3783.028432,4001.611957,44.0,1820.5,2821.0,4373.0,47026.0
ENSG00000067113,1231.0,1604.144598,1363.740198,99.0,783.5,1232.0,1895.5,11477.0
ENSG00000141934,1231.0,5199.349310,3458.617820,19.0,2904.5,4446.0,6755.5,30872.0
ENSG00000162407,1231.0,3497.064175,3169.317648,106.0,1666.0,2643.0,4031.5,29156.0
...,...,...,...,...,...,...,...,...
ENSG00000198648,1231.0,3140.712429,2380.453748,41.0,1572.0,2527.0,4031.0,19071.0
ENSG00000172939,1231.0,3722.880585,1723.384946,670.0,2528.5,3496.0,4589.0,18107.0
ENSG00000138777,1231.0,3702.949634,1659.770835,330.0,2538.5,3444.0,4546.5,11478.0


In [54]:
(x.div(x.sum(axis=1), axis=0) * 1e6).describe().T.describe()

,count,mean,std,min,25%,50%,75%,max
count,4383.0,4383.000000,4383.000000,4383.000000,4383.000000,4383.000000,4383.000000,4383.000000
mean,1231.0,228.154232,175.792219,23.581977,131.406051,188.327797,273.155990,2293.925586
std,0.0,699.890829,544.229416,73.212584,388.711423,587.115184,901.327486,6997.793942
min,1231.0,2.844757,1.834568,0.000000,0.000000,0.000000,0.000000,15.710451
25%,1231.0,30.869035,28.494868,0.537142,11.505474,19.903738,34.806180,343.225429
50%,1231.0,90.994944,65.339059,5.408849,47.486717,70.790567,106.381791,759.399502
75%,1231.0,218.947796,144.402083,21.884184,127.607499,180.433301,255.124164,1832.192596
max,1231.0,18119.603987,16108.832552,2122.829774,10302.640529,14167.575113,25325.128722,203181.340032


In [35]:
np.log(handle_log_zeros(x)).describe().T.describe()

,count,mean,std,min,25%,50%,75%,max
count,4383.0,4383.000000,4383.000000,4383.000000,4383.000000,4383.000000,4383.000000,4383.000000
mean,1231.0,6.973990,0.813468,3.828616,6.467146,6.987237,7.498350,9.761330
std,0.0,1.732251,0.395634,2.242965,1.924030,1.769260,1.613516,1.342836
min,1231.0,0.250655,0.337514,0.000000,0.000000,0.000000,0.000000,5.700444
25%,1231.0,5.946977,0.530560,2.197225,5.408290,5.985194,6.572282,8.855022
50%,1231.0,7.228039,0.676984,4.262680,6.811244,7.253470,7.700521,9.676148
75%,1231.0,8.162357,0.975016,5.575949,7.791522,8.195885,8.580262,10.582764
max,1231.0,12.493520,3.370230,10.065394,12.202454,12.528200,13.168221,15.631073
